In [ ]:
from pathlib import Path

def _sionna_repo_root():
    """Find project root when Jupyter is started from root or notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "scenes" / "University.xml").is_file():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory")


from sionna.rt import load_scene

scene = load_scene(str(_sionna_repo_root() / "scenes" / "University.xml"))
print("scene loaded")
print("num objects =", len(scene.objects))
print(scene.objects.keys())

In [ ]:
scene.preview()

In [ ]:
import os
import numpy as np
import mitsuba as mi

try:
    mi.set_variant("cuda_ad_mono_polarized")
except Exception:
    mi.set_variant("llvm_ad_mono_polarized")

from sionna.rt import (
    load_scene,
    PlanarArray,
    Transmitter,
    Receiver,
    PathSolver,
)

# ============================================================
# 1) Scene paths
# ============================================================
SCENE_NO_RIS   = str(_sionna_repo_root() / "scenes" / "University.xml")
SCENE_WITH_RIS = str(_sionna_repo_root() / "scenes" / "University_RIS.xml")

if not os.path.exists(SCENE_NO_RIS):
    raise FileNotFoundError(f"Scene not found: {SCENE_NO_RIS}")

if not os.path.exists(SCENE_WITH_RIS):
    raise FileNotFoundError(f"Scene not found: {SCENE_WITH_RIS}")

# ============================================================
# 2) Simulation parameters
# ============================================================
FREQ_HZ = 1e9
MAX_DEPTH = 4

# User-defined positions
BS_POS = np.array([-130.0, 15.0, -135.0], dtype=np.float32)
UE_POS = np.array([-130.0, 1.5, -180.0], dtype=np.float32)

# BS: 8x8 planar array, UE: single antenna
BS_ROWS, BS_COLS = 8, 8
UE_ROWS, UE_COLS = 1, 1

# ============================================================
# 2.5) Orientation
# - 아래 값은 BS array를 수직으로 세우기 위한 후보 설정
# - preview로 보고 원하는 방향 아니면 [0, 0, np.pi/2] 등으로 바꿔보면 됨
# ============================================================
BS_ORIENTATION = [0.0, 0.0, 0.0]

# ============================================================
# 3) Utilities
# ============================================================
def db10(x):
    x = np.maximum(np.real(x), 1e-30)
    return 10*np.log10(x)

def db20(x):
    x = np.maximum(np.abs(x), 1e-30)
    return 20*np.log10(x)

def setup_scene(scene_path, bs_pos, ue_pos):
    scene = load_scene(scene_path)
    scene.frequency = FREQ_HZ

    # BS transmit array
    scene.tx_array = PlanarArray(
        num_rows=BS_ROWS,
        num_cols=BS_COLS,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V"
    )

    # UE receive array
    scene.rx_array = PlanarArray(
        num_rows=UE_ROWS,
        num_cols=UE_COLS,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V"
    )

    # Add BS and UE
    scene.add(
        Transmitter(
            name="bs",
            position=bs_pos,
            orientation=BS_ORIENTATION
        )
    )
    scene.add(Receiver(name="ue", position=ue_pos))

    return scene

def compute_paths(scene):
    solver = PathSolver()
    paths = solver(
        scene,
        max_depth=MAX_DEPTH,
        los=True,
        specular_reflection=True,
        diffraction=True,
        edge_diffraction=False,
        refraction=False,
        diffuse_reflection=False
    )
    return paths

def get_narrowband_channel(paths):
    freqs = np.array([FREQ_HZ], dtype=np.float32)
    h = np.array(paths.cfr(frequencies=freqs, out_type="numpy"))

    print("CFR shape:", h.shape)

    if h.ndim != 6:
        raise RuntimeError(f"Unexpected CFR shape: {h.shape}")

    H = h[0, :, 0, :, 0, 0]
    print("Extracted H shape:", H.shape)
    return H

def svd_beamforming(H):
    U, S, Vh = np.linalg.svd(H)
    w_rx = U[:, 0]
    w_tx = Vh.conj().T[:, 0]
    sigma_max = S[0]
    return w_tx, w_rx, sigma_max

def metrics_from_channel(H, w_tx, w_rx):
    power_no_bf = np.linalg.norm(H, "fro")**2
    h_eff = np.vdot(w_rx, H @ w_tx)
    power_bf = np.abs(h_eff)**2
    return power_no_bf, power_bf, h_eff

def count_paths(paths):
    try:
        a, tau = paths.cir(out_type="numpy")
        if a.ndim >= 5:
            return a.shape[4]
        return None
    except Exception:
        return None

def evaluate_scene(scene_path, label):
    print(f"\n==============================")
    print(f"Evaluating: {label}")
    print(f"Scene file : {scene_path}")
    print(f"==============================")

    scene = setup_scene(scene_path, BS_POS, UE_POS)
    paths = compute_paths(scene)
    H = get_narrowband_channel(paths)

    w_tx, w_rx, sigma_max = svd_beamforming(H)
    p_no_bf, p_bf, h_eff = metrics_from_channel(H, w_tx, w_rx)
    n_paths = count_paths(paths)

    return {
        "label": label,
        "scene": scene,
        "paths": paths,
        "H": H,
        "w_tx": w_tx,
        "w_rx": w_rx,
        "sigma_max": sigma_max,
        "power_no_bf": p_no_bf,
        "power_bf": p_bf,
        "h_eff": h_eff,
        "num_paths": n_paths,
    }

# ============================================================
# 4) Run both cases
# ============================================================
res_no_ris = evaluate_scene(SCENE_NO_RIS, "No RIS")
res_ris    = evaluate_scene(SCENE_WITH_RIS, "With RIS")

# ============================================================
# 5) Print comparison
# ============================================================
for res in [res_no_ris, res_ris]:
    print(f"\n=== {res['label']} ===")
    print("H shape               :", res["H"].shape)
    print("num_paths             :", res["num_paths"])
    print("sigma_max             :", res["sigma_max"])
    print("||H||_F^2 [dB]        :", db10(res["power_no_bf"]))
    print("|h_eff|^2 [dB]        :", db10(res["power_bf"]))
    print("|h_eff| [dB]          :", db20(res["h_eff"]))
    print("Beamforming gain [dB] :", db10(res["power_bf"]) - db10(res["power_no_bf"]))

ris_gain_bf_db = db10(res_ris["power_bf"]) - db10(res_no_ris["power_bf"])
ris_gain_nb_db = db10(res_ris["power_no_bf"]) - db10(res_no_ris["power_no_bf"])

print("\n======================================")
print("Comparison: With RIS vs No RIS")
print("======================================")
print("No-BF power improvement [dB] :", ris_gain_nb_db)
print("BF power improvement    [dB] :", ris_gain_bf_db)

# ============================================================
# 6) Optional preview
# ============================================================
# res_no_ris["scene"].preview()
res_ris["scene"].preview()

In [ ]:
import sionna
import sionna.rt as rt

print("Sionna version:", sionna.__version__)